In [1]:
import pandas as pd

In [16]:
resume = pd.read_csv('UpdatedResumeDataSet.csv')
jobs = pd.read_csv('nyc-jobs.csv')

In [27]:
jobs.shape


(2946, 28)

In [11]:
import pandas as pd
import numpy as np
from datetime import datetime
import re

def clean_nyc_jobs_data(df):
    """
    Clean NYC jobs dataset for ontology generation
    """

    print("🧹 Cleaning NYC Jobs Dataset")
    print("=" * 40)

    # Create a copy to avoid modifying original
    cleaned_df = df.copy()

    # 1. Clean column names (remove spaces and special characters)
    print("📝 Cleaning column names...")
    cleaned_df.columns = cleaned_df.columns.str.replace(' ', '_')
    cleaned_df.columns = cleaned_df.columns.str.replace('/', '_')
    cleaned_df.columns = cleaned_df.columns.str.replace('#', 'Number')
    cleaned_df.columns = cleaned_df.columns.str.replace('-', '_')

    # 2. Handle missing values strategically
    print("🔧 Handling missing values...")

    # Fill Job Category with 'General' if missing
    cleaned_df['Job_Category'] = cleaned_df['Job_Category'].fillna('General')

    # Fill Part-Time indicator with 'Unknown' if missing
    cleaned_df['Full_Time_Part_Time_indicator'] = cleaned_df['Full_Time_Part_Time_indicator'].fillna('Unknown')

    # Fill Preferred Skills with 'Not Specified'
    cleaned_df['Preferred_Skills'] = cleaned_df['Preferred_Skills'].fillna('Not Specified')

    # Fill Additional Information with empty string
    cleaned_df['Additional_Information'] = cleaned_df['Additional_Information'].fillna('')

    # 3. Create salary range column
    print("💰 Processing salary information...")
    cleaned_df['Salary_Range'] = cleaned_df['Salary_Range_From'].astype(str) + ' - ' + cleaned_df['Salary_Range_To'].astype(str)
    cleaned_df['Salary_Average'] = (cleaned_df['Salary_Range_From'] + cleaned_df['Salary_Range_To']) / 2

    # 4. Clean text fields
    print("📄 Cleaning text fields...")

    # Clean job descriptions (remove HTML, extra spaces)
    cleaned_df['Job_Description'] = cleaned_df['Job_Description'].astype(str)
    cleaned_df['Job_Description'] = cleaned_df['Job_Description'].str.replace(r'<[^>]*>', '', regex=True)  # Remove HTML
    cleaned_df['Job_Description'] = cleaned_df['Job_Description'].str.replace(r'\s+', ' ', regex=True)  # Multiple spaces to single
    cleaned_df['Job_Description'] = cleaned_df['Job_Description'].str.strip()

    # Clean minimum qualifications
    cleaned_df['Minimum_Qual_Requirements'] = cleaned_df['Minimum_Qual_Requirements'].astype(str)
    cleaned_df['Minimum_Qual_Requirements'] = cleaned_df['Minimum_Qual_Requirements'].str.replace(r'<[^>]*>', '', regex=True)
    cleaned_df['Minimum_Qual_Requirements'] = cleaned_df['Minimum_Qual_Requirements'].str.replace(r'\s+', ' ', regex=True)

    # Clean preferred skills
    cleaned_df['Preferred_Skills'] = cleaned_df['Preferred_Skills'].astype(str)
    cleaned_df['Preferred_Skills'] = cleaned_df['Preferred_Skills'].str.replace(r'<[^>]*>', '', regex=True)
    cleaned_df['Preferred_Skills'] = cleaned_df['Preferred_Skills'].str.replace(r'\s+', ' ', regex=True)

    # 5. Standardize categorical values
    print("🔄 Standardizing categorical values...")

    # Standardize posting type
    cleaned_df['Posting_Type'] = cleaned_df['Posting_Type'].str.title()

    # Standardize full-time/part-time
    cleaned_df['Full_Time_Part_Time_indicator'] = cleaned_df['Full_Time_Part_Time_indicator'].str.upper()

    # 6. Create derived columns for ontology
    print("🏗️ Creating derived columns...")

    # Experience level (inferred from job title and description)
    def extract_experience_level(title, description):
        title_desc = f"{title} {description}".lower()

        if any(word in title_desc for word in ['senior', 'sr.', 'lead', 'principal', 'director', 'manager']):
            return 'Senior'
        elif any(word in title_desc for word in ['junior', 'jr.', 'entry', 'assistant', 'trainee']):
            return 'Junior'
        elif any(word in title_desc for word in ['associate', 'specialist', 'analyst']):
            return 'Mid-Level'
        else:
            return 'General'

    cleaned_df['Experience_Level'] = cleaned_df.apply(
        lambda row: extract_experience_level(row['Business_Title'], row['Job_Description']),
        axis=1
    )

    # Skills extracted from job description and preferred skills
    def extract_skills(description, preferred_skills):
        text = f"{description} {preferred_skills}".lower()

        # Common skills to look for
        skills = []
        skill_keywords = [
            'python', 'java', 'javascript', 'sql', 'excel', 'powerpoint', 'word',
            'project management', 'communication', 'leadership', 'analysis',
            'customer service', 'data analysis', 'microsoft office', 'budgeting',
            'planning', 'coordination', 'supervision', 'training', 'research',
            'writing', 'reporting', 'database', 'computer', 'software'
        ]

        for skill in skill_keywords:
            if skill in text:
                skills.append(skill)

        return '; '.join(skills) if skills else 'General skills'

    cleaned_df['Extracted_Skills'] = cleaned_df.apply(
        lambda row: extract_skills(row['Job_Description'], row['Preferred_Skills']),
        axis=1
    )

    # 7. Create location categories
    print("📍 Processing location information...")

    # Simplify work location
    def categorize_location(location):
        location = str(location).lower()
        if 'manhattan' in location:
            return 'Manhattan'
        elif 'brooklyn' in location:
            return 'Brooklyn'
        elif 'queens' in location:
            return 'Queens'
        elif 'bronx' in location:
            return 'Bronx'
        elif 'staten island' in location:
            return 'Staten Island'
        else:
            return 'Multiple/Other'

    cleaned_df['Location_Category'] = cleaned_df['Work_Location'].apply(categorize_location)

    # 8. Remove duplicates and invalid rows
    print("🔍 Removing duplicates and invalid data...")

    # Remove exact duplicates
    initial_count = len(cleaned_df)
    cleaned_df = cleaned_df.drop_duplicates()
    print(f"   Removed {initial_count - len(cleaned_df)} duplicate rows")

    # Remove rows with invalid salaries
    cleaned_df = cleaned_df[cleaned_df['Salary_Range_From'] > 0]
    cleaned_df = cleaned_df[cleaned_df['Salary_Range_To'] > 0]

    # 9. Select key columns for ontology generation
    print("📊 Selecting key columns for ontology...")

    key_columns = [
        'Job_ID', 'Agency', 'Business_Title', 'Civil_Service_Title',
        'Job_Category', 'Experience_Level', 'Salary_Average', 'Salary_Range',
        'Full_Time_Part_Time_indicator', 'Location_Category', 'Work_Location',
        'Job_Description', 'Minimum_Qual_Requirements', 'Preferred_Skills',
        'Extracted_Skills', 'Number_Of_Positions', 'Level'
    ]

    ontology_df = cleaned_df[key_columns].copy()

    # 10. Final data quality report
    print("\n📋 Data Quality Report:")
    print(f"   Original records: {len(df)}")
    print(f"   Cleaned records: {len(ontology_df)}")
    print(f"   Missing values per column:")

    for col in ontology_df.columns:
        missing = ontology_df[col].isnull().sum()
        if missing > 0:
            print(f"     {col}: {missing} ({missing/len(ontology_df)*100:.1f}%)")

    print(f"\n🎯 Key Statistics:")
    print(f"   Job categories: {ontology_df['Job_Category'].nunique()}")
    print(f"   Agencies: {ontology_df['Agency'].nunique()}")
    print(f"   Location categories: {ontology_df['Location_Category'].nunique()}")
    print(f"   Experience levels: {ontology_df['Experience_Level'].nunique()}")
    print(f"   Salary range: ${ontology_df['Salary_Average'].min():.0f} - ${ontology_df['Salary_Average'].max():.0f}")

    return ontology_df


In [12]:
cleaned_df = clean_nyc_jobs_data(jobs)


🧹 Cleaning NYC Jobs Dataset
📝 Cleaning column names...
🔧 Handling missing values...
💰 Processing salary information...
📄 Cleaning text fields...
🔄 Standardizing categorical values...
🏗️ Creating derived columns...
📍 Processing location information...
🔍 Removing duplicates and invalid data...
   Removed 31 duplicate rows
📊 Selecting key columns for ontology...

📋 Data Quality Report:
   Original records: 2946
   Cleaned records: 2899
   Missing values per column:

🎯 Key Statistics:
   Job categories: 130
   Agencies: 52
   Location categories: 5
   Experience levels: 4
   Salary range: $10 - $218587


In [15]:
cleaned_df.shape

(2899, 17)

In [22]:
# resume.unique() 
# show unique Categories in resume
resume_categories = resume['Category'].unique()


In [26]:
jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2946 entries, 0 to 2945
Data columns (total 28 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Job ID                         2946 non-null   int64  
 1   Agency                         2946 non-null   object 
 2   Posting Type                   2946 non-null   object 
 3   # Of Positions                 2946 non-null   int64  
 4   Business Title                 2946 non-null   object 
 5   Civil Service Title            2946 non-null   object 
 6   Title Code No                  2946 non-null   object 
 7   Level                          2946 non-null   object 
 8   Job Category                   2944 non-null   object 
 9   Full-Time/Part-Time indicator  2751 non-null   object 
 10  Salary Range From              2946 non-null   float64
 11  Salary Range To                2946 non-null   float64
 12  Salary Frequency               2946 non-null   o

In [4]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.chunk import ne_chunk

# Download required NLTK data (run once)
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('stopwords')

# =============================================================================
# CORPUS-BASED ONTOLOGY DEVELOPMENT
# =============================================================================

class CorpusBasedOntologyGenerator:
    def __init__(self, min_frequency=5, max_concepts=50):
        self.min_frequency = min_frequency
        self.max_concepts = max_concepts
        self.stop_words = set(stopwords.words('english'))
        self.entities = {}
        self.relationships = []
        self.concepts = {}

    def extract_concepts_from_corpus(self, text_corpus, corpus_name):
        """Extract concepts using NLP techniques"""

        print(f"🔍 Extracting concepts from {corpus_name} corpus...")

        # Clean and preprocess text
        cleaned_corpus = []
        for text in text_corpus:
            if pd.notna(text):
                # Remove HTML, special chars, normalize
                clean_text = re.sub(r'<[^>]+>', '', str(text))
                clean_text = re.sub(r'[^\w\s]', ' ', clean_text)
                clean_text = ' '.join(clean_text.split())
                cleaned_corpus.append(clean_text.lower())

        # Extract key terms using TF-IDF
        vectorizer = TfidfVectorizer(
            max_features=1000,
            stop_words='english',
            ngram_range=(1, 3),
            min_df=self.min_frequency
        )

        tfidf_matrix = vectorizer.fit_transform(cleaned_corpus)
        feature_names = vectorizer.get_feature_names_out()

        # Get top terms by TF-IDF score
        tfidf_scores = tfidf_matrix.sum(axis=0).A1
        top_terms = [(feature_names[i], tfidf_scores[i])
                    for i in tfidf_scores.argsort()[-self.max_concepts:][::-1]]

        # Extract named entities
        entities = self._extract_named_entities(cleaned_corpus)

        # Extract noun phrases (potential concepts)
        noun_phrases = self._extract_noun_phrases(cleaned_corpus)

        # Topic modeling for concept discovery
        topics = self._discover_topics(cleaned_corpus)

        return {
            'top_terms': top_terms,
            'entities': entities,
            'noun_phrases': noun_phrases,
            'topics': topics,
            'corpus_name': corpus_name
        }

    def _extract_named_entities(self, corpus):
        """Extract named entities using NLTK"""
        entities = Counter()

        for text in corpus[:100]:  # Sample for performance
            try:
                tokens = word_tokenize(text)
                pos_tags = pos_tag(tokens)
                named_entities = ne_chunk(pos_tags)

                for chunk in named_entities:
                    if hasattr(chunk, 'label'):
                        entity = ' '.join([token for token, pos in chunk.leaves()])
                        entities[entity.lower()] += 1
            except:
                continue

        return entities.most_common(20)

    def _extract_noun_phrases(self, corpus):
        """Extract noun phrases as potential concepts"""
        noun_phrases = Counter()

        for text in corpus[:200]:  # Sample for performance
            try:
                tokens = word_tokenize(text)
                pos_tags = pos_tag(tokens)

                # Simple noun phrase extraction
                current_phrase = []
                for word, pos in pos_tags:
                    if pos.startswith('NN') or pos.startswith('JJ'):
                        current_phrase.append(word.lower())
                    else:
                        if len(current_phrase) > 1:
                            phrase = ' '.join(current_phrase)
                            if len(phrase) > 3:  # Filter short phrases
                                noun_phrases[phrase] += 1
                        current_phrase = []
            except:
                continue

        return noun_phrases.most_common(30)

    def _discover_topics(self, corpus):
        """Discover topics using LDA"""
        try:
            vectorizer = TfidfVectorizer(
                max_features=500,
                stop_words='english',
                min_df=3
            )

            doc_term_matrix = vectorizer.fit_transform(corpus)

            lda = LatentDirichletAllocation(
                n_components=5,  # Number of topics
                random_state=42,
                max_iter=10
            )

            lda.fit(doc_term_matrix)

            # Extract topics
            feature_names = vectorizer.get_feature_names_out()
            topics = []

            for topic_idx, topic in enumerate(lda.components_):
                top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]
                topics.append({
                    'topic_id': topic_idx,
                    'words': top_words,
                    'weight': topic.max()
                })

            return topics
        except:
            return []

    def infer_entity_types(self, concepts):
        """Automatically infer entity types from concepts"""

        print("🧠 Inferring entity types from concepts...")

        # Classification patterns
        patterns = {
            'Person': ['person', 'candidate', 'employee', 'worker', 'staff', 'individual'],
            'Organization': ['agency', 'department', 'company', 'organization', 'bureau', 'office'],
            'Role': ['position', 'job', 'title', 'role', 'post', 'appointment'],
            'Skill': ['skill', 'ability', 'knowledge', 'expertise', 'competency', 'experience'],
            'Location': ['location', 'place', 'site', 'area', 'region', 'office', 'building'],
            'Process': ['process', 'procedure', 'method', 'system', 'operation', 'function'],
            'Document': ['document', 'form', 'report', 'application', 'certificate', 'license'],
            'Requirement': ['requirement', 'qualification', 'criteria', 'standard', 'condition']
        }

        inferred_entities = defaultdict(list)

        # Analyze all concepts
        all_concepts = []
        for concept_data in concepts.values():
            all_concepts.extend([term for term, score in concept_data['top_terms']])
            all_concepts.extend([entity for entity, count in concept_data['entities']])
            all_concepts.extend([phrase for phrase, count in concept_data['noun_phrases']])

        # Classify concepts
        for concept in set(all_concepts):
            concept_lower = concept.lower()
            classified = False

            for entity_type, keywords in patterns.items():
                if any(keyword in concept_lower for keyword in keywords):
                    inferred_entities[entity_type].append(concept)
                    classified = True
                    break

            if not classified:
                # Try to infer from context
                if len(concept.split()) == 1 and concept.isalpha():
                    inferred_entities['Attribute'].append(concept)
                elif 'ing' in concept_lower:
                    inferred_entities['Process'].append(concept)
                else:
                    inferred_entities['General'].append(concept)

        return dict(inferred_entities)

    def discover_relationships(self, resume_concepts, job_concepts):
        """Discover relationships through co-occurrence analysis"""

        print("🔗 Discovering relationships through co-occurrence...")

        relationships = []

        # Find common concepts between datasets
        resume_terms = set([term for term, score in resume_concepts['top_terms']])
        job_terms = set([term for term, score in job_concepts['top_terms']])

        common_terms = resume_terms & job_terms

        # Infer relationships based on co-occurrence
        for term in common_terms:
            relationships.append(('Person', 'hasAttribute', term))
            relationships.append(('Role', 'requires', term))

        # Relationship patterns from context
        resume_entities = [entity for entity, count in resume_concepts['entities']]
        job_entities = [entity for entity, count in job_concepts['entities']]

        # Basic relationship inference
        relationships.extend([
            ('Person', 'appliesFor', 'Role'),
            ('Role', 'postedBy', 'Organization'),
            ('Role', 'locatedAt', 'Location'),
            ('Person', 'possesses', 'Skill'),
            ('Role', 'demands', 'Skill'),
            ('Organization', 'offers', 'Role'),
            ('Role', 'hasRequirement', 'Requirement')
        ])

        return relationships

    def generate_ontology_structure(self, resume_df, jobs_df):
        """Generate complete ontology structure"""

        print("🏗️ Generating corpus-based ontology structure...")

        # Extract concepts from resume corpus
        resume_corpus = resume_df['Cleaned_Resume'].dropna().tolist()
        resume_concepts = self.extract_concepts_from_corpus(resume_corpus, 'Resume')

        # Extract concepts from job corpus
        job_text_columns = ['Job_Description', 'Minimum_Qual_Requirements', 'Preferred_Skills']
        job_corpus = []

        for col in job_text_columns:
            if col in jobs_df.columns:
                job_corpus.extend(jobs_df[col].dropna().astype(str).tolist())

        job_concepts = self.extract_concepts_from_corpus(job_corpus, 'Job')

        # Infer entity types
        all_concepts = {
            'resume': resume_concepts,
            'job': job_concepts
        }

        entity_types = self.infer_entity_types(all_concepts)

        # Discover relationships
        relationships = self.discover_relationships(resume_concepts, job_concepts)

        # Create ontology structure
        ontology = {
            'entities': entity_types,
            'relationships': relationships,
            'concepts': {
                'resume_concepts': resume_concepts,
                'job_concepts': job_concepts
            },
            'metadata': {
                'approach': 'corpus_based',
                'resume_documents': len(resume_df),
                'job_documents': len(jobs_df),
                'total_entities': len(entity_types),
                'total_relationships': len(relationships)
            }
        }

        return ontology

# =============================================================================
# TASK-BASED ONTOLOGY DEVELOPMENT
# =============================================================================

class TaskBasedOntologyGenerator:
    def __init__(self, task_focus='job_matching'):
        self.task_focus = task_focus

    def analyze_task_requirements(self, resume_df, jobs_df):
        """Analyze what entities and relationships are needed for the task"""

        print("🎯 Analyzing task requirements for job matching...")

        # For job matching task, we need:
        task_entities = {}
        task_relationships = []

        # Analyze what columns exist in data
        resume_columns = resume_df.columns.tolist()
        job_columns = jobs_df.columns.tolist()

        print(f"Resume columns: {resume_columns}")
        print(f"Job columns: {job_columns}")

        # Infer entities from data structure
        if 'Category' in resume_columns:
            task_entities['PersonCategory'] = {
                'instances': resume_df['Category'].unique().tolist(),
                'count': resume_df['Category'].nunique(),
                'source': 'resume_categories'
            }

        if 'Job_Category' in job_columns:
            task_entities['JobCategory'] = {
                'instances': jobs_df['Job_Category'].dropna().unique().tolist(),
                'count': jobs_df['Job_Category'].nunique(),
                'source': 'job_categories'
            }

        if 'Agency' in job_columns:
            task_entities['Agency'] = {
                'instances': jobs_df['Agency'].unique().tolist(),
                'count': jobs_df['Agency'].nunique(),
                'source': 'job_agencies'
            }

        if 'Work_Location' in job_columns:
            task_entities['Location'] = {
                'instances': jobs_df['Work_Location'].unique().tolist(),
                'count': jobs_df['Work_Location'].nunique(),
                'source': 'job_locations'
            }

        # Salary analysis
        if 'Salary_Range_From' in job_columns and 'Salary_Range_To' in job_columns:
            salary_ranges = jobs_df[['Salary_Range_From', 'Salary_Range_To']].dropna()
            task_entities['SalaryRange'] = {
                'instances': f"{len(salary_ranges)} ranges",
                'count': len(salary_ranges),
                'source': 'job_salaries'
            }

        # Task-specific relationships
        task_relationships = [
            ('Person', 'appliesTo', 'Job'),
            ('Person', 'belongsTo', 'PersonCategory'),
            ('Job', 'belongsTo', 'JobCategory'),
            ('Job', 'offeredBy', 'Agency'),
            ('Job', 'locatedAt', 'Location'),
            ('Job', 'offers', 'SalaryRange')
        ]

        return {
            'entities': task_entities,
            'relationships': task_relationships,
            'task_focus': self.task_focus,
            'metadata': {
                'approach': 'task_based',
                'entities_count': len(task_entities),
                'relationships_count': len(task_relationships)
            }
        }


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/superfunguy/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/superfunguy/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /Users/superfunguy/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package words to
[nltk_data]     /Users/superfunguy/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/superfunguy/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:

# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    # Load your data
    resume_df = pd.read_csv('UpdatedResumeDataSet.csv')
    jobs_df = pd.read_csv('nyc-jobs.csv')

    # Clean job column names
    jobs_df.columns = jobs_df.columns.str.replace(' ', '_').str.replace('/', '_').str.replace('#', 'Number')

    print("📊 Data Overview:")
    print(f"Resumes: {len(resume_df)}")
    print(f"Jobs: {len(jobs_df)}")

    # CORPUS-BASED APPROACH
    print("\n" + "="*50)
    print("CORPUS-BASED ONTOLOGY GENERATION")
    print("="*50)

    corpus_generator = CorpusBasedOntologyGenerator(min_frequency=5, max_concepts=30)
    corpus_ontology = corpus_generator.generate_ontology_structure(resume_df, jobs_df)

    # TASK-BASED APPROACH
    print("\n" + "="*50)
    print("TASK-BASED ONTOLOGY GENERATION")
    print("="*50)

    task_generator = TaskBasedOntologyGenerator(task_focus='job_matching')
    task_ontology = task_generator.analyze_task_requirements(resume_df, jobs_df)

    # RESULTS
    print("\n" + "="*50)
    print("ONTOLOGY COMPARISON")
    print("="*50)

    print(f"\nCorpus-based approach:")
    print(f"  Entities: {len(corpus_ontology['entities'])}")
    print(f"  Relationships: {len(corpus_ontology['relationships'])}")
    print(f"  Entity types: {list(corpus_ontology['entities'].keys())}")

    print(f"\nTask-based approach:")
    print(f"  Entities: {len(task_ontology['entities'])}")
    print(f"  Relationships: {len(task_ontology['relationships'])}")
    print(f"  Entity types: {list(task_ontology['entities'].keys())}")

    # Save results
    import json

    with open('corpus_based_ontology.json', 'w') as f:
        json.dump(corpus_ontology, f, indent=2, default=str)

    with open('task_based_ontology.json', 'w') as f:
        json.dump(task_ontology, f, indent=2, default=str)

    print(f"\n✅ Results saved to JSON files")
    print(f"📁 corpus_based_ontology.json")
    print(f"📁 task_based_ontology.json")

    return corpus_ontology, task_ontology

# Run the analysis
if __name__ == "__main__":
    corpus_ontology, task_ontology = main()

📊 Data Overview:
Resumes: 962
Jobs: 2946

CORPUS-BASED ONTOLOGY GENERATION
🏗️ Generating corpus-based ontology structure...


KeyError: 'Cleaned_Resume'